# README

Companion notebook to reproduce, symbolically, the proofs of Section 2 for the manuscript

> [1] Alexandre d’Aspremont, Frederik Kunstner, Adrien Taylor (2026). "Accelerated gradient methods
in unconstrained convex optimization." To appear in Acta Numerica.

For ease of use, the notebook can be opened in Colab through the following link: 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AccelerationMethods/Acta-Acceleration/blob/main/Section2_symbolics.ipynb)

This notebook requires the use of the SymPy package (https://www.sympy.org/en/index.html).

In [ ]:
!pip install sympy # run if sympy is not installed locally or if run from Google Colab

# SymPy verifications of the main theorems from Section 2

The next cell setups SymPy.


In [2]:
import sympy as sp

def check(expr):
    return sp.simplify(expr)

## 1 Verification of Theorem 2.1

### 1.1 Statement and proof (as in the manuscript)

**Theorem 2.1. (Gradient descent, smooth convex minimization)** Let $f\in\mathcal{F}_{0,L}(\mathbb{R}^d)$. Consider a specific iteration $k\geq 0$ at $x_k\in\mathbb{R}^d$, with $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$, and let $A_k\geq0$. Then,
$$
A_{k+1}\big(f(x_{k+1})-f_\star\big)+\tfrac{L}{2}\|x_{k+1}-x_\star\|^2 \leqslant A_k\big(f(x_k)-f_\star\big)+\tfrac{L}{2}\|x_{k}-x_\star\|^2
$$
with $A_{k+1}=A_k+1$.

*Proof.* Define $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac{L}{2}\|x_{k}-x_\star\|^2$. The proof consists of a weighted sum of the following inequalities:

- convexity between $x_\star$ and $x_k$, with weight $\lambda_1=A_{k+1}-A_k=1$:

  $$
  f_\star\geq f(x_k)+\langle\nabla f(x_k),\,x_\star-x_k\rangle,
  $$

- smoothness between $x_{k+1}$ and $x_k$ via the descent lemma, with weight $\lambda_2=A_{k+1}=A_k+1$:

  $$
  f(x_{k+1})\leq f(x_k)-\tfrac1{2L}\|\nabla f(x_k)\|^2.
  $$
The weighted sum reads

$$
0\geq \lambda_1\big[f(x_k)-f_\star+\langle\nabla f(x_k),\,x_\star-x_k\rangle\big]
+\lambda_2\big[f(x_{k+1})-f(x_k)+\tfrac1{2L}\|\nabla f(x_k)\|^2\big],
$$

which, using the update $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$, can be reformulated exactly as

$$
0\geq \Phi_{k+1}-\Phi_k+\tfrac{A_k}{2L}\|\nabla f(x_k)\|^2 .
$$

The coefficient of the gradient norm is nonnegative, hence $\Phi_{k+1}\leq\Phi_k$. $\square$


### 1.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [3]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')

xk = sp.symbols('x_k')
xs = sp.symbols('x_\star')
gk = sp.symbols('\\nabla{}f(x_k)')
gkp1 = sp.symbols('\\nabla{}f(x_{k+1})')
fk = sp.symbols('f(x_k)')
fkp1 = sp.symbols('f(x_{k+1})')
fs = sp.symbols('f_\star')

Akp1 = Ak + 1
xkp1 = xk - 1/L * gk

Now, one can write the two inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [4]:
inequality1 = fk - fs + gk * (xs - xk)
inequality2 = fkp1 - fk + 1 / 2 / L * gk ** 2
lambda1 = 1
lambda2 = Akp1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2

Similarly, one can define the two potential values and form the target inequality

In [5]:
potentialk = Ak * (fk - fs) + L / 2 * (xk - xs) ** 2
potentialkp1 = Akp1 * (fkp1 - fs) + L / 2 * (xkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk + Ak / 2 / L * gk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [6]:
check(weighted_sum - target_inequality)

0

## 3 Verification of Theorem 2.3

### 3.1 Statement and proof (as in the manuscript)

**Theorem 2.3. (Conjugate gradients, $m=0$)** Let $f\in\mathcal{F}_{0,L}(\mathbb{R}^d)$, let $k\geq0$, $A_k,a_k\geq0$, and $x_k,z_k,y_k\in\mathbb{R}^d$ satisfying
$$
\langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0.
$$
Define the next points $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$, $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$. If $a_k^2-a_k-A_k\leq0$, then it holds that
$$
A_{k+1}\big(f(x_{k+1})-f_\star\big)+\tfrac{L}{2}\|z_{k+1}-x_\star\|^2 \leqslant A_k\big(f(x_k)-f_\star\big)+\tfrac{L}{2}\|z_{k}-x_\star\|^2
$$
with $A_{k+1}=A_k+a_k$.

*Proof.* Define $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac{L}{2}\|z_{k}-x_\star\|^2$. The proof consists of a weighted sum of the following inequalities:

- convexity between $x_\star$ and $y_k$, with weight $\lambda_1=a_k$:

  $$
  f_\star\geq f(y_k)+\langle\nabla f(y_k),\,x_\star-y_k\rangle,
  $$

- convexity between $x_k$ and $y_k$, with weight $\lambda_2=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- smoothness between $x_{k+1}$ and $y_k$ via the descent lemma, with weight $\lambda_3=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2,
  $$

- the relation satisfied by $x_k,z_k,y_k$, with weight $\lambda_4=1$:

  $$
  \langle\nabla f(y_k),\,A_kx_k+a_kz_k-A_{k+1}y_k\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f_\star+\langle\nabla f(y_k),\,x_\star-y_k\rangle\big]\\
&+\lambda_2\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_3\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_4\langle\nabla f(y_k),\,A_{k+1}y_k-A_kx_k-a_kz_k\rangle ,
\end{aligned}
$$

which, using the updates $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and $z_{k+1}=z_k-\tfrac{a_k}{L}\nabla f(y_k)$, together with $A_{k+1}=A_k+a_k$, can be reformulated exactly as

$$
0\geq\Phi_{k+1}-\Phi_k+\tfrac{A_{k+1}-a_k^2}{2L}\|\nabla f(y_k)\|^2 ,
$$

using similar steps as before. Then, the claim follows from the last inequality as soon as $A_{k+1}\geq a_k^2$, thereby concluding the proof. $\square$

### 3.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [7]:
L = sp.symbols('L')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
xs = sp.symbols('x_\star')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
fs = sp.symbols('f_\star')

Akp1 = Ak + ak
xkp1 = yk - 1/L * gyk
zkp1 = zk - ak/L * gyk

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [8]:
inequality1 = fyk - fs + gyk * (xs - yk)
inequality2 = fyk - fxk + gyk * (xk - yk)
inequality3 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality4 = - gyk * (Ak * xk + ak * zk - Akp1 * yk)
lambda1 = ak
lambda2 = Ak
lambda3 = Akp1
lambda4 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3 + lambda4 * inequality4

Similarly, one can define the two potential values and form the target inequality

In [9]:
potentialk = Ak * (fxk - fs) + L / 2 * (zk - xs) ** 2
potentialkp1 = Akp1 * (fxkp1 - fs) + L / 2 * (zkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk  + (Akp1 - ak ** 2) / 2 / L * gyk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [10]:
check(weighted_sum - target_inequality)

0

## 4 Verification of Theorem 2.7

### 4.1 Statement and proof (as in the manuscript)

**Theorem 2.7. (Gradient descent, smooth strongly convex minimization)** For $f\in\mathcal{F}_{m,L}(\mathbb{R}^d)$ with $0\leq m<L$, let $k\geq0$, with $A_k\geq 0$, $x_k\in\mathbb{R}^d$, and $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$. Then:
$$
A_{k+1}\big(f(x_{k+1})-f_\star\big)+\tfrac{L+m A_{k+1}}{2}\|x_{k+1}-x_\star\|^2 \leqslant A_k\big(f(x_k)-f_\star\big)+\tfrac{L+m A_k}{2}\|x_{k}-x_\star\|^2
$$
with $A_{k+1}=(1+A_k)/(1-q)$.

*Proof.* Define $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac{L+m A_k}{2}\|x_{k}-x_\star\|^2$. The proof consists of a weighted sum of the two inequalities of the proof of Theorem 2.1, the first one now carrying its quadratic term:

- the lower bound between $x_\star$ and $x_k$, with weight $\lambda_1=A_{k+1}-A_k$:

  $$
  f_\star\geq f(x_k)+\langle\nabla f(x_k),\,x_\star-x_k\rangle+\tfrac m2\|x_\star-x_k\|^2,
  $$

- the descent lemma, with weight $\lambda_2=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(x_k)-\tfrac1{2L}\|\nabla f(x_k)\|^2 .
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(x_k)-f_\star+\langle\nabla f(x_k),\,x_\star-x_k\rangle+\tfrac m2\|x_\star-x_k\|^2\big]\\
&+\lambda_2\big[f(x_{k+1})-f(x_k)+\tfrac1{2L}\|\nabla f(x_k)\|^2\big] ,
\end{aligned}
$$

which, using the update $x_{k+1}=x_k-\tfrac1L\nabla f(x_k)$ and $A_{k+1}=\tfrac{1+A_k}{1-q}$, can be reformulated exactly as

$$
0\geq\Phi_{k+1}-\Phi_k+\tfrac{A_k}{2L}\|\nabla f(x_k)\|^2 ,
$$

hence $\Phi_{k+1}\leq\Phi_k$, as in the convex case. $\square$


### 4.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [11]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')

xk = sp.symbols('x_k')
xs = sp.symbols('x_\star')
gk = sp.symbols('\\nabla{}f(x_k)')
gkp1 = sp.symbols('\\nabla{}f(x_{k+1})')
fk = sp.symbols('f(x_k)')
fkp1 = sp.symbols('f(x_{k+1})')
fs = sp.symbols('f_\star')

q = m / L
Akp1 = (Ak + 1) / (1 - q)
xkp1 = xk - 1/L * gk

Now, one can write the two inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [12]:
inequality1 = fk - fs + gk * (xs - xk) + m / 2 * (xs - xk) ** 2
inequality2 = fkp1 - fk + 1 / 2 / L * gk ** 2
lambda1 = Akp1 - Ak
lambda2 = Akp1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2

Similarly, one can define the two potential values and form the target inequality

In [13]:
potentialk = Ak * (fk - fs) + (L + m * Ak) / 2 * (xk - xs) ** 2
potentialkp1 = Akp1 * (fkp1 - fs) + (L + m * Akp1)/ 2 * (xkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk + Ak / 2 / L * gk ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [14]:
check(weighted_sum - target_inequality)

0

## 5 Verification of Theorem 2.9

### 5.1 Statement and proof (as in the manuscript)

**Theorem 2.9. (Conjugate gradients, $m\geq0$)** Let $f\in\mathcal{F}_{m,L}(\mathbb{R}^d)$ with $0\leq m<L$, $q=m/L$, let $k\geq0$, $A_k,a_k\geq0$, and $x_k,z_k,y_k\in\mathbb{R}^d$ satisfying the relation
$$
\langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0,
$$
with $\delta_k=\tfrac{a_k}{1+qA_{k+1}}$. Define the next points $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and
$$
z_{k+1}=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),
$$
If $A_{k+1}(1+qA_{k+1})-a_k^2\geq 0$, then it holds that
$$
A_{k+1}\big(f(x_{k+1})-f_\star\big)+\tfrac{L+m A_{k+1}}{2}\|z_{k+1}-x_\star\|^2 \leqslant A_k\big(f(x_k)-f_\star\big)+\tfrac{L+mA_k}{2}\|z_{k}-x_\star\|^2
$$
with $A_{k+1}=A_k+a_k$.

*Proof.* Define $\Phi_k=A_k\big(f(x_k)-f_\star\big)+\tfrac{L+mA_k}{2}\|z_{k}-x_\star\|^2$. The proof consists of a weighted sum of the four inequalities of the proof of Theorem 2.3, possibly adapted to carry a strongly convex component, as follows:

- the lower bound between $x_\star$ and $y_k$, with weight $\lambda_1=a_k$:

  $$
  f_\star\geq f(y_k)+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac m2\|x_\star-y_k\|^2,
  $$

- convexity between $x_k$ and $y_k$, with weight $\lambda_2=A_k$:

  $$
  f(x_k)\geq f(y_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle,
  $$

- the descent lemma, with weight $\lambda_3=A_{k+1}$:

  $$
  f(x_{k+1})\leq f(y_k)-\tfrac{1}{2L}\|\nabla f(y_k)\|^2 ,
  $$

- the relation satisfied by $x_k,z_k,y_k$, with weight $\lambda_4=1$:

  $$
  \langle\nabla f(y_k),\,A_k(x_k-y_k)+\delta_k(1+qA_k)(z_k-y_k)\rangle\geq0.
  $$

The weighted sum reads

$$
\begin{aligned}
0\geq{}&\lambda_1\big[f(y_k)-f_\star+\langle\nabla f(y_k),\,x_\star-y_k\rangle+\tfrac m2\|x_\star-y_k\|^2\big]\\
&+\lambda_2\big[f(y_k)-f(x_k)+\langle\nabla f(y_k),\,x_k-y_k\rangle\big]\\
&+\lambda_3\big[f(x_{k+1})-f(y_k)+\tfrac{1}{2L}\|\nabla f(y_k)\|^2\big]\\
&+\lambda_4\langle\nabla f(y_k),\,A_k(y_k-x_k)+\delta_k(1+qA_k)(y_k-z_k)\rangle,
\end{aligned}
$$

which, using the updates $x_{k+1}=y_k-\tfrac1L\nabla f(y_k)$ and $z_{k+1}=z_k+q\delta_k\,(y_k-z_k)-\tfrac{\delta_k}{L}\nabla f(y_k),$, can be reformulated exactly as

$$
0\geq\Phi_{k+1}-\Phi_k+\tfrac{A_{k+1}-a_k\delta_k}{2L}\|\nabla f(y_k)\|^2+\delta_k(1+qA_k)\,\tfrac m2\|y_k-z_k\|^2 ,
$$

which allows us to reach the claim using the fact that both ${A_{k+1}-a_k\delta_k}\geq 0$ (equivalent to our assumption $A_{k+1}(1+qA_{k+1})-a_k^2\geq 0$) and $\delta_k(1+qA_k)\,\tfrac m2\geq 0$, thereby concluding the proof. $\square$

### 5.2 SymPy verification

To verify the expression symbolically, one can define the following symbols:

In [15]:
L = sp.symbols('L')
m = sp.symbols('m')
Ak = sp.symbols('A_k')
ak = sp.symbols('a_k')

xk = sp.symbols('x_k')
zk = sp.symbols('z_k')
yk = sp.symbols('y_k')
xs = sp.symbols('x_\star')
gyk = sp.symbols('\\nabla{}f(y_k)')
fyk = sp.symbols('f(y_k)')
fxk = sp.symbols('f(x_k)')
fxkp1 = sp.symbols('f(x_{k+1})')
fs = sp.symbols('f_\star')

Akp1 = Ak + ak
deltak = ak / (1 + q * Akp1)
xkp1 = yk - 1/L * gyk
zkp1 = zk + q * deltak * (yk - zk) - deltak / L * gyk

Now, as before, one can write the inequalities used in the proof (in the form "$\ldots\leqslant 0$") and their weights to form the weighted sum:

In [16]:
inequality1 = fyk - fs + gyk * (xs - yk) + m / 2 * (xs - yk) ** 2
inequality2 = fyk - fxk + gyk * (xk - yk)
inequality3 = fxkp1 - fyk + 1 / 2 / L * gyk ** 2
inequality4 = gyk * (Ak * (yk - xk) + deltak * (1 + q * Ak) * (yk - zk))
lambda1 = ak
lambda2 = Ak
lambda3 = Akp1
lambda4 = 1

weighted_sum = lambda1 * inequality1 + lambda2 * inequality2 + lambda3 * inequality3 + lambda4 * inequality4

Similarly, one can define the two potential values and form the target inequality

In [17]:
potentialk = Ak * (fxk - fs) + (L + m * Ak) / 2 * (zk - xs) ** 2
potentialkp1 = Akp1 * (fxkp1 - fs) + (L + m * Akp1) / 2 * (zkp1 - xs) ** 2

target_inequality = potentialkp1 - potentialk  + (Akp1 - ak * deltak) / 2 / L * gyk ** 2 +  deltak * (1 + q * Ak) * m / 2 * (yk - zk) ** 2

It remains to check that the expressions match, which we check by verifying their difference is identically zero

In [18]:
check(weighted_sum - target_inequality)

0